# MyTravels — Preview Runbook

Runs the full MyTravels stack via Docker Compose. No local SDK or build tools required.

| Service | Purpose | Port(s) |
|---------|---------|--------|
| PostgreSQL | Primary database | 5432 |
| RabbitMQ | Message broker | 5672 · 15672 (UI) |
| MinIO | Object storage | 9000 (API) · 9090 (Console) |

> Run each cell in order.

## Prerequisites

| Tool | Purpose | Install |
|------|---------|---------|
| Rancher Desktop | Docker engine + `docker compose` | `brew install --cask rancher` |
| JupyterLab | Run this notebook | `brew install jupyterlab` |

Once Rancher Desktop is installed, open it and ensure the container engine is running before continuing.

- A `.env` file must exist in this directory (copy from `.env.example` if one exists)

---
## 1. Load environment variables

In [9]:
from pathlib import Path
import os

for line in Path(".env").read_text().splitlines():
    line = line.strip()
    if line and not line.startswith("#") and "=" in line:
        key, _, value = line.partition("=")
        os.environ[key.strip()] = value.strip()

print("Loaded environment from .env")

Loaded environment from .env


---
## 2. Start the stack

In [10]:
%%bash
docker compose up -d
echo "Stack started."

time="2026-05-24T18:53:56+02:00" level=warning msg="No services to build"
 Network 1-starter_project_default Creating 
 Network 1-starter_project_default Created 
 Volume 1-starter_project_pgdata Creating 
 Volume 1-starter_project_pgdata Created 
 Volume 1-starter_project_minio-data Creating 
 Volume 1-starter_project_minio-data Created 
 Volume 1-starter_project_minio-config Creating 
 Volume 1-starter_project_minio-config Created 
 Volume 1-starter_project_mqdata Creating 
 Volume 1-starter_project_mqdata Created 
 Volume 1-starter_project_mqconfig Creating 
 Volume 1-starter_project_mqconfig Created 
 Container mytravels-postgres Creating 
 Container minio Creating 
 Container mytravels-rabbitmq Creating 
 Container mytravels-postgres Created 
 Container cleanup-migrations Creating 
 Container 1-starter_project-migrate-core-db-1 Creating 
 Container cleanup-migrations Created 
 Container 1-starter_project-migrate-core-db-1 Created 
 Container mytravels-rabbitmq Created 
 Container 

Stack started.


---
## 3. Check service health

In [12]:
%%bash
docker compose ps

NAME                                  IMAGE                                        COMMAND                  SERVICE           CREATED          STATUS                    PORTS
1-starter_project-migrate-core-db-1   tshepontlhokoa/mytravels-migrations:v1.0.0   "sh -c 'dotnet tool …"   migrate-core-db   28 seconds ago   Up 17 seconds             
minio                                 quay.io/minio/minio                          "/usr/bin/docker-ent…"   minio             28 seconds ago   Up 27 seconds (healthy)   0.0.0.0:9000->9000/tcp, [::]:9000->9000/tcp, 0.0.0.0:9090->9090/tcp, [::]:9090->9090/tcp
mytravels-postgres                    postgres:17.6-alpine                         "docker-entrypoint.s…"   postgres          28 seconds ago   Up 27 seconds (healthy)   0.0.0.0:5432->5432/tcp, [::]:5432->5432/tcp
mytravels-rabbitmq                    rabbitmq:3-management                        "docker-entrypoint.s…"   rabbitmq          28 seconds ago   Up 27 seconds (healthy)   4369/tcp, 5671/

**Service UIs:**
- RabbitMQ Management: http://localhost:15672
- MinIO Console: http://localhost:9090

---
## 4. View logs (optional)

In [13]:
%%bash
echo "=== minio ===" && docker compose logs --tail=20 minio
echo "=== rabbitmq ===" && docker compose logs --tail=20 rabbitmq
echo "=== postgres ===" && docker compose logs --tail=20 postgres
echo "=== cleanup-migrations ===" && docker compose logs --tail=20 cleanup-migrations
echo "=== migrate-core-db ===" && docker compose logs --tail=20 migrate-core-db

=== minio ===
minio  | INFO: Formatting 1st pool, 1 set(s), 1 drives per set.
minio  | INFO: WARNING: Host local has more than 0 drives of set. A host failure will result in data becoming unavailable.
minio  | MinIO Object Storage Server
minio  | Copyright: 2015-2026 MinIO, Inc.
minio  | License: GNU AGPLv3 - https://www.gnu.org/licenses/agpl-3.0.html
minio  | Version: RELEASE.2025-09-07T16-13-09Z (go1.24.6 linux/arm64)
minio  | 
minio  | API: http://172.22.0.3:9000  http://127.0.0.1:9000 
minio  | WebUI: http://172.22.0.3:9090 http://127.0.0.1:9090  
minio  | 
minio  | Docs: https://docs.min.io
=== rabbitmq ===
mytravels-rabbitmq  | 2026-05-24 16:53:58.965111+00:00 [warning] <0.705.0> By default, this feature can still be used for now.
mytravels-rabbitmq  | 2026-05-24 16:53:58.965111+00:00 [warning] <0.705.0> Its use will not be permitted by default in a future minor RabbitMQ version and the feature will be removed from a future major RabbitMQ version; actual versions to be determined

## 5. Teardown

In [15]:
%%bash
# Stop and remove containers AND volumes — wipes all data
docker compose down --volumes
echo "Containers and volumes removed."

Containers and volumes removed.
